In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score, roc_curve)
import xgboost as xgb
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

**1: Load all sensor data from multiple sheets**

In [ ]:
def load_sensor_data(file_path, sensor_name):
    """
    Load all sheets from an Excel file, extract time and data_value,
    return a single DataFrame with columns: time, sensor_name
    """
    xl = pd.ExcelFile(file_path)
    all_data = []
    for sheet in xl.sheet_names:
        df = pd.read_excel(xl, sheet_name=sheet)
        # Keep only relevant columns
        if 'time' in df.columns and 'data_value' in df.columns:
            df_sub = df[['time', 'data_value']].copy()
            df_sub['sensor'] = sensor_name
            all_data.append(df_sub)
    if not all_data:
        raise ValueError(f"No valid sheets found in {file_path}")
    return pd.concat(all_data, ignore_index=True)

# Load each sensor
cng_df = load_sensor_data('cng_sensor.xlsx', 'cng')
co_df = load_sensor_data('co_sensor.xlsx', 'co')
flame_df = load_sensor_data('flame_sensor.xlsx', 'flame')
lpg_df   = load_sensor_data('lpg_sensor.xlsx', 'lpg')
smoke_df = load_sensor_data('smoke_sensor.xlsx', 'smoke')

print(f"CNG: {len(cng_df)} rows")
print(f"CO: {len(co_df)} rows")
print(f"Flame: {len(flame_df)} rows")
print(f"LPG:   {len(lpg_df)} rows")
print(f"Smoke: {len(smoke_df)} rows")

**2: Pivot to wide format by merging on nearest timestamp**

In [ ]:
# Convert time to datetime and sort
for df in [cng_df, co_df, flame_df, lpg_df, smoke_df]:
    df['time'] = pd.to_datetime(df['time'])
    df.sort_values('time', inplace=True)

# Rename columns for clarity
cng_df.rename(columns={'data_value': 'cng'}, inplace=True)
co_df.rename(columns={'data_value': 'co'}, inplace=True)
flame_df.rename(columns={'data_value': 'flame'}, inplace=True)
lpg_df.rename(columns={'data_value': 'lpg'}, inplace=True)
smoke_df.rename(columns={'data_value': 'smoke'}, inplace=True)

# Start with the most frequent sensor (smoke) as base
base = smoke_df[['time', 'smoke']].copy()
base.sort_values('time', inplace=True)

# Merge flame (nearest)
base = pd.merge_asof(base, flame_df[['time', 'flame']], on='time', direction='nearest')

# Merge lpg (nearest)
base = pd.merge_asof(base, lpg_df[['time', 'lpg']], on='time', direction='nearest')
base = pd.merge_asof(base, cng_df[['time', 'cng']], on='time', direction='nearest')
base = pd.merge_asof(base, co_df[['time', 'co']], on='time', direction='nearest')

# Drop rows where any sensor is missing (optional)
base.dropna(subset=['cng', 'co', 'flame', 'lpg', 'smoke'], inplace=True)

print(f"Merged dataset shape: {base.shape}")
base.head()

**- Display Combined Data**

In [ ]:
# ============================================
# Display the combined wide-format table
# ============================================

print("="*60)
print("COMBINED DATASET (WIDE FORMAT) - BEFORE PREPROCESSING")
print("="*60)

# Show first 10 rows
print("\nFirst 10 rows:")
print(base.head(10))

# Show dataset info
print("\nDataset Info:")
print(base.info())

# Show basic statistics
print("\nBasic Statistics:")
print(base[['flame', 'lpg', 'smoke']].describe())

# Check for missing values
print("\nMissing values per column:")
print(base.isnull().sum())

# Show the distribution of fire events (after adding target in next step)
# But we can already show the distribution using the thresholds
base_temp = base.copy()
base_temp['fire'] = ((base_temp['smoke'] > 300) |
                     (base_temp['lpg'] > 2100) |
                     (base_temp['flame'] > 0.8)).astype(int)

print("\nFire event distribution (using thresholds):")
print(base_temp['fire'].value_counts())
print(f"Percentage of fire events: {base_temp['fire'].mean()*100:.2f}%")

# Show a sample of fire events (where fire=1)
print("\nSample rows where fire = 1 (first 5 occurrences):")
print(base_temp[base_temp['fire'] == 1].head())

# Show a sample of non-fire events
print("\nSample rows where fire = 0 (first 5 occurrences):")
print(base_temp[base_temp['fire'] == 0].head())

# Optional: Plot time series of sensors to visualize fire events
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

axes[0].plot(base['time'], base['smoke'], color='blue', alpha=0.7)
axes[0].axhline(y=300, color='red', linestyle='--', label='Fire threshold (300 ppm)')
axes[0].set_ylabel('Smoke (ppm)')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(base['time'], base['lpg'], color='green', alpha=0.7)
axes[1].axhline(y=2100, color='red', linestyle='--', label='Fire threshold (2100 ppm)')
axes[1].set_ylabel('LPG (ppm)')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(base['time'], base['flame'], color='orange', alpha=0.7)
axes[2].axhline(y=0.8, color='red', linestyle='--', label='Fire threshold (0.8)')
axes[2].set_ylabel('Flame (raw)')
axes[2].set_xlabel('Time')
axes[2].legend()
axes[2].grid(True)

plt.suptitle('Sensor Readings Over Time (Fire events highlighted)')
plt.tight_layout()
plt.show()

**3: Add ID and prepare features/target**

In [ ]:
# Sort by time and add sequential ID
base.sort_values('time', inplace=True)
base.reset_index(drop=True, inplace=True)
base['id'] = base.index + 1   # ID starting from 1

# Define target label (fire detection) using thresholds from the article
# CNG > 1000 CO > 50 Smoke > 300 ppm OR LPG > 2100 ppm OR Flame > 0.8 (raw value)
base['fire'] = ((base['cng'] > 1000) |
                (base['co'] > 50) |
                (base['smoke'] > 300) |
                (base['lpg'] > 2100) |
                (base['flame'] > 0.8)).astype(int)

print("Class distribution:")
print(base['fire'].value_counts())

# Features and target
features = ['cng', 'co', 'flame', 'lpg', 'smoke']
X = base[features].copy()
y = base['fire'].copy()

# Check imbalance ratio
print(f"\nImbalance ratio (0/1): {y.value_counts()[0]/y.value_counts()[1]:.2f}")

**- Export Pre-Processed data**

In [ ]:
# ============================================
# Export pre-processed dataset to CSV
# ============================================

# The DataFrame 'base' already contains:
# - id: sequential integer ID (replaces timestamp)
# - flame, lpg, smoke: sensor readings
# - fire: binary target (1 = fire detected, 0 = normal)

# Save to CSV file
csv_filename = 'preprocessed_fire_dataset.csv'
base[['id', 'flame', 'lpg', 'smoke', 'fire']].to_csv(csv_filename, index=False)

print(f"Pre-processed dataset saved as '{csv_filename}'")
print(f"Total samples: {len(base)}")
print(f"Features: flame, lpg, smoke")
print(f"Target: fire (1 = fire event, 0 = normal)")
print(f"Class distribution:\n{base['fire'].value_counts()}")

# Display first 10 rows of the saved dataset
print("\nFirst 10 rows of the pre-processed dataset:")
print(base[['id', 'flame', 'lpg', 'smoke', 'fire']].head(10))

# Optional: Download the file in Colab
from google.colab import files
files.download(csv_filename)

**4: Train/validation/test split (stratified)**

In [ ]:
# First split: 70% train, 30% temp (which will be split into 15% val, 15% test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Split temp into 15% val, 15% test (relative to original)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train size: {X_train.shape[0]} (fire: {y_train.sum()})")
print(f"Val size:   {X_val.shape[0]}   (fire: {y_val.sum()})")
print(f"Test size:  {X_test.shape[0]}  (fire: {y_test.sum()})")

**- Use Cross-Validation**

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42)

# Use original features and labels (no SMOTE inside CV)
scores = cross_val_score(rf, X, y, cv=cv, scoring='f1')
print(f"Cross-validated F1 scores: {scores}")
print(f"Mean F1: {scores.mean():.4f} (+/- {scores.std():.4f})")

**5: Apply SMOTE on training data**

In [ ]:
# Scale features before SMOTE (SVM and LR need scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE only on training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE - Train size: {X_train_resampled.shape[0]} (fire: {y_train_resampled.sum()})")

**- Perform Time-Based Split**

In [ ]:
# Assuming 'base' is sorted by time
split_idx = int(len(base) * 0.85)  # 85% early data for training, 15% last day for testing
X_train_time = X.iloc[:split_idx]
y_train_time = y.iloc[:split_idx]
X_test_time = X.iloc[split_idx:]
y_test_time = y.iloc[split_idx:]

# Apply SMOTE only on training
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_time, y_train_time)

# Train Random Forest and evaluate on the last day's data
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_resampled, y_train_resampled)
y_pred_time = rf.predict(X_test_time)
print(classification_report(y_test_time, y_pred_time))

**6: Define models and training function**

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42)
}

def train_evaluate(model, X_train, y_train, X_val, y_val, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1] if hasattr(model, "predict_proba") else None

    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred)
    rec = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    auc = roc_auc_score(y_val, y_proba) if y_proba is not None else 0

    return {
        'model': model_name,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'roc_auc': auc
    }

results = []
for name, model in models.items():
    print(f"Training {name}...")
    metrics = train_evaluate(model, X_train_resampled, y_train_resampled, X_val_scaled, y_val, name)
    results.append(metrics)

**7: Compare Model Performance on Validation set**

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('f1_score', ascending=False)
print("Validation Performance Comparison:")
print(results_df.to_string(index=False))

# Plot bar comparison
plt.figure(figsize=(10,6))
metrics_plot = results_df.melt(id_vars='model', value_vars=['accuracy','precision','recall','f1_score','roc_auc'])
sns.barplot(data=metrics_plot, x='model', y='value', hue='variable')
plt.title('Model Performance on Validation Set')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**8: Select Best model and Evaluate on Test set**

In [ ]:
best_model_name = results_df.iloc[0]['model']
best_model = models[best_model_name]
best_model.fit(X_train_resampled, y_train_resampled)

# Test evaluation
y_test_pred = best_model.predict(X_test_scaled)
y_test_proba = best_model.predict_proba(X_test_scaled)[:, 1]

test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred)
test_rec = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)
test_auc = roc_auc_score(y_test, y_test_proba)

print(f"\nBest Model: {best_model_name}")
print(f"Test Accuracy:  {test_acc:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall:    {test_rec:.4f}")
print(f"Test F1-Score:  {test_f1:.4f}")
print(f"Test ROC-AUC:   {test_auc:.4f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {best_model_name} on Test Set')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

**9: Feature Importance for Tree-Based Models**

In [ ]:
if best_model_name in ['Random Forest', 'XGBoost']:
    importances = best_model.feature_importances_
    feat_imp = pd.Series(importances, index=features).sort_values(ascending=False)
    plt.figure(figsize=(8,4))
    feat_imp.plot(kind='bar')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.tight_layout()
    plt.show()

**10: Save Results to CSV**

In [ ]:
# Save all metrics to a CSV file
results_df.to_csv('model_comparison.csv', index=False)
print("Model comparison saved to 'model_comparison.csv'")

# Also save test results
test_results = pd.DataFrame([{
    'best_model': best_model_name,
    'test_accuracy': test_acc,
    'test_precision': test_prec,
    'test_recall': test_rec,
    'test_f1': test_f1,
    'test_auc': test_auc
}])
test_results.to_csv('test_results.csv', index=False)

**11: Export Model**

In [ ]:
import joblib
joblib.dump(best_model, 'fire_detection_rf.pkl')
joblib.dump(scaler, 'scaler.pkl')